In [1]:
from pathlib import Path
from pprint import pprint
from contextlib import suppress
from dioptra.client import connect_json_dioptra_client, select_files_in_directory

DIOPTRA_REST_API_ADDRESS = "http://localhost"
DIOPTRA_REST_API_USER = "user"
DIOPTRA_REST_API_PASS = "pass"


client = connect_json_dioptra_client(DIOPTRA_REST_API_ADDRESS)

with suppress(Exception): client.users.create(DIOPTRA_REST_API_USER, email=f"{DIOPTRA_REST_API_USER}@localhost", password=DIOPTRA_REST_API_PASS)
client.auth.login(DIOPTRA_REST_API_USER, DIOPTRA_REST_API_PASS)

import logging
logging.basicConfig(
    level=logging.DEBUG
)

In [2]:
response = client.workflows.import_resources(group_id=1,
                                             source=select_files_in_directory("extra/", recursive=True),
                                             config_path="dioptra.toml",
                                             resolve_name_conflicts_strategy="overwrite",
                                            )

DEBUG:dioptra.client.sessions:Request made: url=http://localhost/api/v1/workflows/resourceImport  method=POST
DEBUG:urllib3.connectionpool:http://localhost:80 "POST /api/v1/workflows/resourceImport HTTP/1.1" 200 452
DEBUG:dioptra.client.sessions:Response received: url=http://localhost/api/v1/workflows/resourceImport  method=POST  status_code=200


In [3]:
evasion = response['resources']['entrypoints']['Evasion Attack']
evasion_snapshot_id = client.entrypoints.get_by_id(entrypoint_id=evasion)['snapshot']

DEBUG:dioptra.client.sessions:Request made: url=http://localhost/api/v1/entrypoints/2471  method=GET
DEBUG:urllib3.connectionpool:http://localhost:80 "GET /api/v1/entrypoints/2471 HTTP/1.1" 200 21503
DEBUG:dioptra.client.sessions:Response received: url=http://localhost/api/v1/entrypoints/2471  method=GET  status_code=200


In [4]:
response = client.entrypoints.snapshots.task_graph_global_params(
    entrypoint_id=evasion,
    entrypoint_snapshot_id=evasion_snapshot_id, 
    swaps={
        'attack' : 'fast_gradient_method'
    }
)
response

DEBUG:dioptra.client.sessions:Request made: url=http://localhost/api/v1/entrypoints/2471/snapshots/3466/dynamicGlobalParameters  method=GET
DEBUG:urllib3.connectionpool:http://localhost:80 "GET /api/v1/entrypoints/2471/snapshots/3466/dynamicGlobalParameters?swaps=attack%3Dfast_gradient_method HTTP/1.1" 200 352
DEBUG:dioptra.client.sessions:Response received: url=http://localhost/api/v1/entrypoints/2471/snapshots/3466/dynamicGlobalParameters  method=GET  status_code=200


{'entrypointParams': ['batch_size',
  'fgm_eps',
  'normalize_val',
  'seed',
  'dataset_name',
  'dataset_split',
  'model_artifact',
  'fgm_eps_step',
  'image_size',
  'fgm_norm',
  'fgm_minimal',
  'dataset_dir',
  'fgm_target'],
 'topologicalSort': ['random_seed',
  'dataset',
  'adversarial_dataset',
  'predictions',
  'metrics'],
 'activePlugins': [{'name': 'dioptra_optic'}]}